In [ ]:
# CELL 1: Install
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub==0.23.4",
    "diffusers==0.28.2",
    "transformers==4.40.2",
    "accelerate==0.30.1",
    "peft==0.11.1",
    "safetensors==0.4.3",
    "gradio==4.44.0",
    "pyngrok==7.1.6",
])
print("✅ Installed")
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

✅ Installed


{'status': 'ok', 'restart': True}

In [ ]:
# CELL 2: GPU Check
import torch
assert torch.cuda.is_available(), 'No GPU'
print(f'✅ {torch.cuda.get_device_name(0)}')

✅ Tesla T4


In [ ]:
# CELL 3: Load Model
import torch, gc, json, tempfile, os
from diffusers import StableDiffusionXLPipeline, AutoencoderKL, DPMSolverMultistepScheduler
from peft import PeftModel
from huggingface_hub import hf_hub_download

SDXL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
VAE_ID = "madebyollin/sdxl-vae-fp16-fix"
LORA_ID = "YoussefSouissi/celeba-lora"
DEVICE = "cuda"
DTYPE = torch.float16

gc.collect()
torch.cuda.empty_cache()

print("Loading VAE...")
vae = AutoencoderKL.from_pretrained(VAE_ID, torch_dtype=DTYPE)

print("Loading SDXL...")
pipe = StableDiffusionXLPipeline.from_pretrained(
    SDXL_ID, vae=vae, torch_dtype=DTYPE, variant="fp16",
    use_safetensors=True, low_cpu_mem_usage=True
)

print("Loading LoRA...")
config_path = hf_hub_download(repo_id=LORA_ID, filename="adapter_config.json")
weights_path = hf_hub_download(repo_id=LORA_ID, filename="adapter_model.safetensors")

with open(config_path) as f:
    cfg = json.load(f)

for field in ["corda_config","eva_config","qalora_group_size","lora_bias",
              "target_parameters","trainable_token_indices","use_qalora","exclude_modules"]:
    cfg.pop(field, None)

tmp = tempfile.mkdtemp()
with open(os.path.join(tmp, "adapter_config.json"), "w") as f:
    json.dump(cfg, f)
os.symlink(weights_path, os.path.join(tmp, "adapter_model.safetensors"))

pipe.unet = PeftModel.from_pretrained(pipe.unet, tmp, is_trainable=False)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True
)
pipe.to(DEVICE)
pipe.enable_attention_slicing(1)
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()

gc.collect()
torch.cuda.empty_cache()
print("✅ Model ready")

/usr/local/lib/python3.12/dist-packages/diffusers/models/transformers/transformer_2d.py:34: FutureWarning: `Transformer2DModelOutput` is deprecated and will be removed in version 1.0.0. Importing `Transformer2DModelOutput` from `diffusers.models.transformer_2d` is deprecated and this will be removed in a future version. Please use `from diffusers.models.modeling_outputs import Transformer2DModelOutput`, instead.
  deprecate("Transformer2DModelOutput", "1.0.0", deprecation_message)


Loading VAE...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading SDXL...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading LoRA...
✅ Model ready


In [ ]:
# CELL 4: Generation Function
import os, hashlib, time, gc
from PIL import Image

NEGATIVE = "deformed face, mutation, blurry, low quality, ugly, disfigured, watermark, text"

def make_seed(p):
    return int(hashlib.sha256(f"{p}{time.time_ns()}{os.urandom(8).hex()}".encode()).hexdigest(), 16) % (2**31)

def generate(prompt):
    """Generate portrait from text"""
    if not prompt or not prompt.strip():
        return Image.new('RGB', (512, 512), 'white')
    try:
        with torch.no_grad():
            result = pipe(
                prompt.strip(),
                negative_prompt=NEGATIVE,
                generator=torch.Generator(device=DEVICE).manual_seed(make_seed(prompt)),
                num_inference_steps=50,
                guidance_scale=9.0,
                height=1024,
                width=1024,
            )
        gc.collect()
        torch.cuda.empty_cache()
        return result.images[0]
    except Exception as e:
        print(f"Error: {e}")
        return Image.new('RGB', (512, 512), '#ffebee')

print("✅ Ready")

✅ Ready


In [ ]:
# CELL 5: Flask UI - runs directly in notebook
from flask import Flask, render_template_string, request, send_file
from pyngrok import ngrok, conf
from threading import Thread
import torch
import gc
import os
import hashlib
import time
from PIL import Image
from io import BytesIO

# Save references
saved_pipe = pipe
saved_device = DEVICE

NEGATIVE = "deformed face, mutation, blurry, low quality, ugly, disfigured, watermark, text"

def make_seed(p):
    return int(hashlib.sha256(f"{p}{time.time_ns()}{os.urandom(8).hex()}".encode()).hexdigest(), 16) % (2**31)

def generate_image(prompt):
    if not prompt or not prompt.strip():
        return Image.new('RGB', (512, 512), 'white')
    try:
        with torch.no_grad():
            result = saved_pipe(
                prompt.strip(),
                negative_prompt=NEGATIVE,
                generator=torch.Generator(device=saved_device).manual_seed(make_seed(prompt)),
                num_inference_steps=50,
                guidance_scale=9.0,
                height=1024,
                width=1024,
            )
        gc.collect()
        torch.cuda.empty_cache()
        return result.images[0]
    except Exception as e:
        print(f"Error: {e}")
        return Image.new('RGB', (512, 512), '#ffebee')

HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>Portrait Studio</title>
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <style>
        * {margin: 0; padding: 0; box-sizing: border-box;}
        body {font-family: -apple-system, sans-serif; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); min-height: 100vh; padding: 20px;}
        .container {max-width: 800px; margin: 0 auto; background: white; border-radius: 16px; padding: 40px; box-shadow: 0 20px 60px rgba(0,0,0,0.3);}
        h1 {text-align: center; color: #667eea; font-size: 2.5em; margin-bottom: 10px;}
        .subtitle {text-align: center; color: #666; margin-bottom: 30px;}
        .info {background: #f0f4ff; padding: 15px; border-radius: 8px; margin-bottom: 25px; border-left: 4px solid #667eea;}
        textarea {width: 100%; padding: 15px; border: 2px solid #e0e0e0; border-radius: 8px; font-size: 16px; resize: vertical; min-height: 100px; margin-bottom: 15px;}
        textarea:focus {outline: none; border-color: #667eea;}
        button {width: 100%; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border: none; padding: 15px; font-size: 16px; font-weight: 600; border-radius: 8px; cursor: pointer;}
        button:hover {transform: translateY(-2px);}
        button:disabled {opacity: 0.6;}
        .examples {display: grid; gap: 10px; margin-bottom: 20px;}
        .example-btn {background: #f5f5f5; color: #333; padding: 10px; font-size: 14px;}
        #result {margin-top: 30px; text-align: center;}
        #result img {max-width: 100%; border-radius: 8px;}
        .loading {text-align: center; padding: 40px; color: #667eea;}
        .spinner {border: 4px solid #f3f3f3; border-top: 4px solid #667eea; border-radius: 50%; width: 40px; height: 40px; animation: spin 1s linear infinite; margin: 20px auto;}
        @keyframes spin {0% {transform: rotate(0deg);} 100% {transform: rotate(360deg);}}
    </style>
</head>
<body>
    <div class="container">
        <h1>🎨 Portrait Studio</h1>
        <p class="subtitle">AI-powered portrait generation using SDXL + LoRA</p>
        <div class="info"><p><strong>💡 Tip:</strong> Describe the person - age, hair, expression, lighting</p><p><strong>⏱️ Time:</strong> ~30-60 seconds</p></div>
        <form id="generateForm">
            <textarea id="prompt" placeholder="portrait photo of a young woman with brown hair, warm smile, soft lighting, 8k"></textarea>
            <div class="examples">
                <button type="button" class="example-btn" onclick="setPrompt('portrait photo of a young woman with brown hair, warm smile, soft lighting, 8k')">Young woman</button>
                <button type="button" class="example-btn" onclick="setPrompt('portrait of an elderly man with grey beard, wise expression, golden hour light')">Elderly man</button>
                <button type="button" class="example-btn" onclick="setPrompt('close-up portrait of a smiling woman with curly hair, natural daylight')">Smiling woman</button>
            </div>
            <button type="submit" id="generateBtn">🎨 Generate Portrait</button>
        </form>
        <div id="result"></div>
    </div>
    <script>
        function setPrompt(text) {document.getElementById('prompt').value = text;}
        document.getElementById('generateForm').onsubmit = async (e) => {
            e.preventDefault();
            const prompt = document.getElementById('prompt').value;
            if (!prompt.trim()) {alert('Please enter a prompt!'); return;}
            const btn = document.getElementById('generateBtn'); const result = document.getElementById('result');
            btn.disabled = true; btn.textContent = '🎨 Generating...';
            result.innerHTML = '<div class="loading"><div class="spinner"></div><p>Generating... (30-60 seconds)</p></div>';
            try {
                const response = await fetch('/generate', {method: 'POST', headers: {'Content-Type': 'application/x-www-form-urlencoded'}, body: 'prompt=' + encodeURIComponent(prompt)});
                if (response.ok) {
                    const blob = await response.blob(); const url = URL.createObjectURL(blob);
                    result.innerHTML = '<img src="' + url + '"><p style="margin-top: 15px;">✅ Done!</p>';
                } else {result.innerHTML = '<p style="color: red;">❌ Failed</p>';}
            } catch (error) {result.innerHTML = '<p style="color: red;">❌ Error: ' + error.message + '</p>';}
            btn.disabled = false; btn.textContent = '🎨 Generate Portrait';
        };
    </script>
</body>
</html>
"""

app = Flask(__name__)

@app.route('/')
def home():
    return render_template_string(HTML)

@app.route('/generate', methods=['POST'])
def generate():
    prompt = request.form.get('prompt', '')
    img = generate_image(prompt)
    buf = BytesIO()
    img.save(buf, format='PNG')
    buf.seek(0)
    return send_file(buf, mimetype='image/png')

# Start Flask in background thread
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

print("Starting Flask server...")
thread = Thread(target=run_flask, daemon=True)
thread.start()

# Wait for Flask to fully start
time.sleep(10)

# Setup ngrok
NGROK_TOKEN = "3Al6V8zgNfUyJX5KvBTXxjPOyu8_C2e2UumMGpe9Cazn6tdZ"
PORT = 5000

conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()

print("Creating ngrok tunnel...")
tunnel = ngrok.connect(PORT, bind_tls=True)

print("=" * 70)
print("🎨 PORTRAIT STUDIO IS LIVE")
print("=" * 70)
print(f"\n📎 PUBLIC URL: {tunnel.public_url}")
print("\n✅ Ready! Click the link to generate portraits")
print("\n" + "=" * 70)

Starting Flask server...
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Creating ngrok tunnel...
🎨 PORTRAIT STUDIO IS LIVE

📎 PUBLIC URL: https://noneloquent-endocrinologic-doloris.ngrok-free.dev

✅ Ready! Click the link to generate portraits

